# Checkpoint 3: Exploratory Data Analysis

Krish: sections 1, 2, plot 3 and 7. Mahi: Twitter distributions and plot 1. Dravya: FakeNewsNet/Facebook distributions and plot 2.

## 0. Load data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.size"] = 12   # readable axis labels

twitter_daily = pd.read_csv("https://raw.githubusercontent.com/LazerLab/twitter-fake-news-replication/master/figure_1/data/daily_counts_exposures.csv")
twitter_sites = pd.read_csv("https://raw.githubusercontent.com/LazerLab/twitter-fake-news-replication/master/coexposure_network/data/total_exposure_counts.csv")

fake = pd.read_csv("https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset/politifact_fake.csv")
real = pd.read_csv("https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset/politifact_real.csv")
fake["label"] = "fake"
real["label"] = "real"
news = pd.concat([fake, real], ignore_index=True)

facebook = pd.read_csv("https://raw.githubusercontent.com/BuzzFeedNews/2016-10-facebook-fact-check/master/data/facebook-fact-check.csv")
ratings = pd.read_csv("https://raw.githubusercontent.com/hauselin/domain-quality-ratings/main/data/domain_pc1.csv")

## 1. Dataset profiles
We used pandas, so `info()` is our version of `glimpse()` and `isna().sum()` is our version of `null_count()`.

### Primary 1a: Twitter daily counts

In [ ]:
print(twitter_daily.shape)
twitter_daily.info()

In [ ]:
twitter_daily.isna().sum()

In [ ]:
twitter_daily.describe()

**What stands out:** No missing values. `date` is text instead of a date, so we fix that in section 2. The busiest days were Nov 8 and 9 (election day and the day after) with about 1.5 million views. `pct_fake` is between 1.9% and 8.3%, average 5.2%, so fake news was a small part of what people saw every day.

### Primary 1b: Twitter website totals

In [ ]:
print(twitter_sites.shape)
twitter_sites.info()

In [ ]:
twitter_sites.isna().sum()

In [ ]:
twitter_sites.describe()

**What stands out:** No missing values or repeated websites. `N` is very skewed: the mean is about 70,800 but the median is only about 7,700, because a few sites like nytimes.com (3.4 million) are way bigger than the rest. We will need a log scale to plot it.

### Primary 2: FakeNewsNet (PolitiFact)

In [ ]:
print(news.shape)
news.info()

In [ ]:
news.isna().sum()

In [ ]:
# number of tweets that shared each article
news["num_tweets"] = news["tweet_ids"].str.split().str.len()
news.describe()

In [ ]:
news.groupby("label")["num_tweets"].describe()

**What stands out:** 255 articles have no tweets and 61 have no URL. `num_tweets` is very skewed (max 29,060). The surprising part: fake articles have a higher median than real ones (102 vs 58) but a lower mean (422 vs 1,022), so whether fake news "spreads more" depends on which one we look at. Also, some titles didn't look like news at all, which we look at in section 2.

## 2. Anomalies

| Dataset | Problem | Rows | What we did |
|---|---|---|---|
| Twitter daily | `date` is text, not a date | 128 | fixed with `pd.to_datetime` |
| Twitter daily | checked for missing days / wrong percentages | 0 | none found |
| Twitter sites | 72 sites don't match the ratings (mostly subdomains like motto.time.com) | 72 | left it, only 1.6% of views |
| FakeNewsNet | same id labeled both fake and real | 4 | dropped |
| FakeNewsNet | not real articles ("Account Suspended", "website is for sale", MSN homepage) | 20 | dropped |
| FakeNewsNet | no tweet ids | 255 | set num_tweets to 0 |
| FakeNewsNet | missing url | 61 | left it, we don't need url right now |
| FakeNewsNet | urls in different formats (no http, web.archive.org links) | 80 + 196 | left it, matters only if we join on url later |
| FakeNewsNet | very big tweet counts (up to 29,060) | a few | kept them, but we use medians and log scale |

In [ ]:
# Anomaly 1: date stored as text -> datetime
twitter_daily["date"] = pd.to_datetime(twitter_daily["date"])
twitter_daily = twitter_daily.sort_values("date")

# Anomaly 2: check for missing days, and that pct_fake matches the counts
all_days = pd.date_range(twitter_daily["date"].min(), twitter_daily["date"].max())
print("missing days:", len(all_days) - twitter_daily["date"].nunique())
check = (twitter_daily["red"] + twitter_daily["orange"] + twitter_daily["black"]) / twitter_daily["N"]
print("rows where pct_fake doesn't match:", ((check - twitter_daily["pct_fake"]).abs() > 0.000001).sum())

In [ ]:
# Anomaly 3: Twitter websites with no quality rating
no_rating = twitter_sites[~twitter_sites["website"].isin(ratings["domain"])]
print(len(no_rating), "sites have no rating")
print("share of all views:", round(no_rating["N"].sum() / twitter_sites["N"].sum() * 100, 1), "%")
no_rating.sort_values("N", ascending=False).head()

In [ ]:
# Anomaly 4: same id labeled both fake and real
dup_ids = news[news["id"].duplicated(keep=False)]
print(len(dup_ids), "rows")
dup_ids[["id", "label", "title"]]

In [ ]:
# Anomaly 5: junk pages that aren't news articles
junk_words = "Account Suspended|under construction|for sale|Outlook, Office, Skype"
junk = news["title"].str.contains(junk_words, case=False)
print(junk.sum(), "junk rows")
news.loc[junk, "title"].value_counts()

In [ ]:
# Cleaning: drop anomalies 4 and 5, fix anomaly 6
news_clean = news[~news["id"].isin(dup_ids["id"]) & ~junk].copy()
news_clean["num_tweets"] = news_clean["num_tweets"].fillna(0)

print("before:", news.shape, " after:", news_clean.shape)
news_clean["label"].value_counts()

In [ ]:
# Anomalies 7 and 8: flagged, not fixed
print("missing url:", news_clean["news_url"].isna().sum())
print("url without http:", (~news_clean["news_url"].dropna().str.startswith("http")).sum())
print("web.archive.org url:", news_clean["news_url"].dropna().str.contains("web.archive.org").sum())

## 3. Distributions

### Twitter (Mahi)
TODO: histograms of `pct_fake` and `N`, describe the shape of each

In [ ]:

# Mahi: pct_fake histogram
plt.figure(figsize=(9, 5))
plt.hist(twitter_daily['pct_fake'], bins=20, edgecolor='black')
plt.title('Distribution of daily fake news share on Twitter')
plt.xlabel('Fraction of news views from fake news sites')
plt.ylabel('Number of days')
plt.show()

In [ ]:
# Mahi: N histogram
plt.figure(figsize=(9, 5))
plt.hist(twitter_daily['N'] / 1000000, bins=20, edgecolor='black')
plt.title('Distribution of total daily Twitter news views')
plt.xlabel('Total news views per day (millions)')
plt.ylabel('Number of days')
plt.show()


### FakeNewsNet and Facebook (Dravya)
TODO: histogram of `num_tweets` (log scale) and bar chart of `Rating` counts, describe the shape of each

In [ ]:
# Dravya: num_tweets histogram


In [ ]:
# Dravya: Facebook Rating counts


## 4. Relationships

### Plot 1 (Mahi): fake news share over time
**pct_fake:** Most days were around 5%, usually between 4% and 6%. The shape is balanced, not skewed. Outliers: Oct 15 was highest (8.3%) and Nov 9 was lowest (1.9%).

**N:** Most days had about 400,000 to 600,000 views. The shape is right-skewed because a few days had way more views. Those were big news days: election day (1.5 million), the day after (1.3 million), and two debate days (about 1 million).

In [ ]:
# Mahi: pct_fake histogram
plt.figure(figsize=(9, 5))
plt.hist(twitter_daily['pct_fake'], bins=20, edgecolor='black')
plt.title('Distribution of daily fake news share on Twitter')
plt.xlabel('Fraction of news views from fake news sites')
plt.ylabel('Number of days')
plt.show()


### Plot 2 (Dravya): tweets for fake vs real articles, compared with Facebook shares
TODO: box plot of num_tweets by label + median share_count by Rating, then so what / expected? / what next

In [ ]:
# Dravya: plot 2


### Plot 3 (Krish): website quality vs Twitter views
Here we join the Twitter website totals with the quality ratings (secondary dataset) on website name.

In [ ]:
joined = twitter_sites.merge(ratings, left_on="website", right_on="domain")
print(joined.shape)

plt.figure(figsize=(9, 6))
plt.scatter(joined["pc1"], joined["N"], alpha=0.5)
plt.yscale("log")
plt.xlabel("Website quality rating (0 = lowest, 1 = highest)")
plt.ylabel("Total Twitter views (log scale)")
plt.title("Website quality vs. Twitter views, 2016 election")
plt.show()

In [ ]:
# group websites into low / medium / high quality and compare
joined["quality"] = pd.cut(joined["pc1"], bins=[0, 0.4, 0.7, 1], labels=["low", "medium", "high"], include_lowest=True)
summary = joined.groupby("quality", observed=True)["N"].agg(["count", "median", "sum"])
summary["share_of_views_%"] = (summary["sum"] / summary["sum"].sum() * 100).round(1)
summary

In [ ]:
# correlation (spearman, because N is very skewed)
print("spearman correlation:", round(joined["pc1"].corr(joined["N"], method="spearman"), 2))

# the low-quality sites that still got lots of views
joined[joined["quality"] == "low"].sort_values("N", ascending=False).head(5)

**So what:** We expected low-quality sites to get more views, but it was the opposite. High-quality sites got 57% of all views and low-quality sites only 11%. The correlation is weak though (0.17), and a few low-quality sites were still huge, like breitbart.com with about 1 million views. So low-quality news was a small part overall but concentrated in a few big sites.

**Expected or unexpected?** Mostly unexpected. We thought "fake news spreads more" would show up here, but total views went mostly to mainstream sites.

**What next:** Total views depends on how many articles a site posts, so next we want to look at views per article. We could also split the sites by political side.

## 5. Revised research questions

1. **Does the typical fake article get shared more than the typical real one?** Changed because the median and mean gave opposite answers, so we need to say which one we mean.
2. **How did the fake news share on Twitter change over time?** because plot 1 showed the share went down on big news days and up on weekends, so the amount of total news seems to matter too.)*
3. **Is low-quality news concentrated in a few websites, and do they get more views per article?** Changed because plot 3 showed low-quality sites get fewer views overall, but a few of them are really big.